# 18. Autoencoder와 VAE — 생성 모델의 시작

> **제18장** · **이론편 대응: 15장 (생성 모델 I)**
> **예상 소요**: 60분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음 (15장에서 설치한 torchvision 사용)
> **다운로드**: FashionMNIST (15장에서 받았다면 재사용)

---

## 이 장에서 하는 일

지금까지는 **입력 → 정답**을 배웠다. 이번에는 **데이터 자체의 구조**를 배운다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | Autoencoder 구현 — 압축과 복원 | 15.1절 |
| 2 | 잠재 공간 시각화 | 15.2절 |
| 3 | **AE의 한계 — 생성이 안 된다** | 15.3절 |
| 4 | **재파라미터화 검증** ★ | 15.4절 |
| 5 | VAE 구현 | 15.4절 |
| 6 | 잠재 공간 보간과 생성 | 15.5절 |
| 7 | 흐릿함의 원인 | 15.6절 |

**3절이 이 장의 전환점이다.** AE로는 왜 새 이미지를 못 만드는지 직접 확인한 뒤,
VAE가 그것을 어떻게 해결하는지 본다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

# 데이터 준비 (15장에서 받은 것을 재사용)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
data_dir = root / "data"

train_data = datasets.FashionMNIST(
    root=str(data_dir), train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(
    root=str(data_dir), train=False, download=True, transform=transforms.ToTensor())

# CPU에서도 빠르게 끝나도록 일부만
N_TRAIN = 6000
train_subset = Subset(train_data, range(N_TRAIN))
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)

print(f"학습 데이터: {N_TRAIN:,}장 (전체 {len(train_data):,}장 중)")
print(f"클래스: {len(train_data.classes)}개")

---

## 1. Autoencoder — 압축과 복원 (이론편 15.1절)

Autoencoder의 구조는 단순하다.

```
입력 (784) → 인코더 → 잠재 벡터 (2) → 디코더 → 출력 (784)
                         ↑
                    병목(bottleneck)
```

**핵심은 가운데를 좁게 만드는 것**이다. 784개 숫자를 2개로 줄였다가 다시 784개로 되돌려야 하므로,
모델은 "무엇이 중요한지" 스스로 골라야 한다.

손실함수는 **입력과 출력의 차이**다. 정답 레이블이 필요 없으므로 비지도학습이다.

$$L = \lVert x - \hat{x} \rVert^2$$

In [ ]:
import torch
import torch.nn as nn


class Autoencoder(nn.Module):
    # 기본 Autoencoder (이론편 15.1절)

    def __init__(self, latent_dim=2):
        super().__init__()
        # 인코더: 784 → 128 → latent
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        # 디코더: latent → 128 → 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid(),          # 픽셀값이 0~1이므로
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z).view(-1, 1, 28, 28)
        return out, z


model_ae = Autoencoder(latent_dim=2).to(device)

print("=" * 55)
print("Autoencoder 구조")
print("=" * 55)
print(f"입력      : 1 x 28 x 28 = 784")
print(f"잠재 차원 : 2            ← 392배 압축")
print(f"출력      : 784")
print()
print(f"파라미터: {sum(p.numel() for p in model_ae.parameters()):,}개")
print()
print("784개 숫자를 2개로 줄였다가 되돌려야 한다.")
print("정보를 대부분 버릴 수밖에 없으므로, 무엇을 남길지가 학습된다.")

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)
model_ae = Autoencoder(latent_dim=2).to(device)
optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3)

EPOCHS = 15
losses_ae = []

print("=" * 55)
print("Autoencoder 학습")
print("=" * 55)
t0 = time.time()

for epoch in range(EPOCHS):
    model_ae.train()
    total = 0.0
    for xb, _ in train_loader:          # 레이블(_)을 쓰지 않는다
        xb = xb.to(device)
        optimizer.zero_grad()
        out, z = model_ae(xb)
        loss = nn.functional.mse_loss(out, xb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)

    avg = total / len(train_subset)
    losses_ae.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f"  에폭 {epoch+1:3}: 손실 {avg:.5f}  ({time.time()-t0:.0f}초)")

print("-" * 55)
print(f"최종 재구성 손실: {losses_ae[-1]:.5f}")
print(f"소요 시간: {time.time()-t0:.0f}초")

In [ ]:
import torch
import matplotlib.pyplot as plt

model_ae.eval()
with torch.no_grad():
    sample = torch.stack([test_data[i][0] for i in range(8)]).to(device)
    recon, z = model_ae(sample)

fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i in range(8):
    axes[0, i].imshow(sample[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_title("원본", loc="left", fontsize=10)
axes[1, 0].set_title("복원 (잠재 2차원 거쳐)", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

print("2개 숫자만 남기고 나머지를 버렸는데도 대략적인 형태가 복원된다.")
print("다만 세부 무늬는 사라졌다 — 2차원으로는 담을 수 없는 정보다.")
print()
print("잠재 벡터 예시 (앞 3개)")
for i in range(3):
    print(f"  {test_data.classes[test_data[i][1]]:<14}→ {z[i].cpu().numpy().round(3)}")

---

## 2. 잠재 공간 시각화 — 이론편 15.2절

잠재 차원을 2로 잡은 이유가 있다. **그대로 그림으로 그릴 수 있기 때문**이다.

각 이미지가 잠재 공간의 어디에 놓이는지 보면, 모델이 무엇을 기준으로 데이터를
정리했는지 알 수 있다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 시험 데이터를 잠재 공간에 투영
model_ae.eval()
zs, labels = [], []
with torch.no_grad():
    for i in range(0, 2000, 256):
        batch = torch.stack([test_data[j][0] for j in range(i, min(i+256, 2000))]).to(device)
        lbl = [test_data[j][1] for j in range(i, min(i+256, 2000))]
        _, z = model_ae(batch)
        zs.append(z.cpu().numpy())
        labels.extend(lbl)

Z = np.concatenate(zs)
labels = np.array(labels)

print("=" * 55)
print("잠재 공간")
print("=" * 55)
print(f"투영한 이미지: {len(Z)}장")
print(f"잠재 벡터 범위: x {Z[:,0].min():.2f}~{Z[:,0].max():.2f}, "
      f"y {Z[:,1].min():.2f}~{Z[:,1].max():.2f}")
print()

fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap="tab10", s=8, alpha=0.6)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.ax.set_yticklabels(train_data.classes, fontsize=8)
ax.set_xlabel("잠재 차원 1")
ax.set_ylabel("잠재 차원 2")
ax.set_title("Autoencoder의 잠재 공간")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("같은 종류끼리 모여 있다.")
print("레이블을 알려주지 않았는데도 스스로 분류에 가까운 구조를 만든 것이다.")
print("→ 이론편 15.2절에서 다룬 '의미 있는 표현의 학습'")

---

## 3. AE의 한계 — 생성이 안 된다 (이론편 15.3절) ★

Autoencoder는 압축과 복원은 잘한다. 그렇다면 **디코더에 아무 값이나 넣으면 새 이미지가 나올까?**

이론편 15.3절에서 "그렇지 않다"고 했다. 직접 확인해 보자.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

model_ae.eval()

# 잠재 공간에서 무작위로 뽑아 디코더에 넣어 본다
rng = np.random.RandomState(0)
z_min, z_max = Z.min(axis=0), Z.max(axis=0)
random_z = rng.uniform(z_min, z_max, size=(8, 2))

with torch.no_grad():
    generated = model_ae.decoder(
        torch.tensor(random_z, dtype=torch.float32).to(device)
    ).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for i in range(8):
    axes[i].imshow(generated[i], cmap="gray")
    axes[i].set_title(f"({random_z[i][0]:.1f}, {random_z[i][1]:.1f})", fontsize=7)
    axes[i].axis("off")
fig.suptitle("잠재 공간에서 무작위로 뽑아 디코딩한 결과", fontsize=12)
plt.tight_layout()
plt.show()

print("일부는 옷 같지만, 알아볼 수 없는 것도 섞여 있다.")
print()
print("왜 그럴까 — 2절 그림을 다시 보자.")
print("  점들이 모여 있는 곳이 있고, 텅 빈 곳이 있다.")
print("  빈 곳의 좌표를 넣으면 모델이 학습한 적 없는 입력이라 엉뚱한 결과가 나온다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 잠재 공간이 얼마나 비어 있는지 확인
print("=" * 55)
print("잠재 공간의 밀도")
print("=" * 55)

# 격자로 나눠 각 칸에 점이 몇 개 있는지 센다
GRID = 12
h, xe, ye = np.histogram2d(Z[:, 0], Z[:, 1], bins=GRID)
empty = (h == 0).sum()

print(f"격자 {GRID}x{GRID} = {GRID*GRID}칸")
print(f"비어 있는 칸: {empty}칸 ({empty/(GRID*GRID)*100:.0f}%)")
print()
print("데이터가 특정 영역에만 몰려 있고 나머지는 비어 있다.")
print("디코더는 점이 있는 곳만 학습했으므로, 빈 곳에 대해서는 보장이 없다.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
im = ax.imshow(h.T, origin="lower", cmap="YlOrRd",
               extent=[xe[0], xe[-1], ye[0], ye[-1]], aspect="auto")
plt.colorbar(im, ax=ax, label="데이터 개수")
ax.set_title("잠재 공간의 데이터 밀도")
ax.set_xlabel("잠재 차원 1")
ax.set_ylabel("잠재 차원 2")

ax = axes[1]
ax.scatter(Z[:, 0], Z[:, 1], s=6, alpha=0.4, color="#64748B", label="실제 데이터")
ax.scatter(random_z[:, 0], random_z[:, 1], s=120, marker="X",
           color="#EA580C", edgecolor="white", linewidth=1.5, label="무작위 샘플")
ax.set_title("무작위 샘플의 위치")
ax.set_xlabel("잠재 차원 1")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("X 표시 중 일부가 점이 없는 곳에 떨어졌다.")
print("이것이 이론편 15.3절에서 말한 'AE로는 생성할 수 없다'는 뜻이다.")
print()
print("해결하려면 잠재 공간이 '빈 곳 없이 채워지도록' 만들어야 한다.")
print("→ VAE의 아이디어")

---

## 4. 재파라미터화 — 이론편 15.4절 값 검증 ★

VAE의 핵심은 **점 하나가 아니라 분포를 학습하는 것**이다. 인코더가 평균 $\mu$와
표준편차 $\sigma$를 내놓고, 거기서 샘플링해 잠재 벡터를 만든다.

그런데 여기에 문제가 있다. **샘플링은 미분할 수 없다.**

이론편 15.4절에서 다룬 해법이 재파라미터화다.

$$z = \mu + \sigma \cdot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, 1)$$

**이론편에서 확인한 값** ($\mu = 2.0$, $\sigma = 0.5$)

| $\epsilon$ | $z$ | $\partial z/\partial\sigma$ |
|---|---|---|
| −1.0 | 1.50 | −1.0 |
| 0.0 | 2.00 | 0.0 |
| 0.5 | 2.25 | 0.5 |
| 1.5 | 2.75 | 1.5 |

In [ ]:
import torch
import numpy as np

print("=" * 60)
print("이론편 15.4절 재파라미터화 검증")
print("=" * 60)

# 이론편과 같은 설정
mu = torch.tensor(2.0, requires_grad=True)
sigma = torch.tensor(0.5, requires_grad=True)
eps = torch.tensor([-1.0, 0.0, 0.5, 1.5])

z = mu + sigma * eps

print(f"mu = {mu.item()}, sigma = {sigma.item()}")
print()
print(f"{'eps':<10}{'z = mu + sigma*eps':<24}{'이론편 값'}")
print("-" * 60)
book_z = [1.50, 2.00, 2.25, 2.75]
for e, zi, b in zip(eps, z.detach(), book_z):
    print(f"{e.item():<10.1f}{zi.item():<24.4f}{b:.2f}")
print("-" * 60)
assert np.allclose(z.detach().numpy(), book_z)
print("[OK] 이론편 15.4절 값과 일치")
print()

# 미분이 되는지 확인 — 이것이 핵심
z.sum().backward()
print("미분 가능성 확인 (z의 합에 대한 그래디언트)")
print(f"  d(sum z)/d(mu)    = {mu.grad.item():.4f}   (기대: 1 x 4개 = 4.0)")
print(f"  d(sum z)/d(sigma) = {sigma.grad.item():.4f}   (기대: eps 합 = {eps.sum().item()})")
assert abs(mu.grad.item() - 4.0) < 1e-6
assert abs(sigma.grad.item() - eps.sum().item()) < 1e-6
print()
print("[OK] mu와 sigma에 대해 미분이 된다 — 역전파로 학습할 수 있다")

### 만약 직접 샘플링했다면

비교를 위해 재파라미터화 없이 바로 뽑아 보자. **미분이 끊긴다.**

In [ ]:
import torch

print("=" * 60)
print("재파라미터화가 없으면")
print("=" * 60)

mu2 = torch.tensor(2.0, requires_grad=True)
sigma2 = torch.tensor(0.5, requires_grad=True)

# 방식 A: 분포에서 직접 뽑기
dist = torch.distributions.Normal(mu2, sigma2)
z_direct = dist.sample()          # sample()은 그래프를 끊는다
print(f"[직접 샘플링] z = {z_direct.item():.4f}")
print(f"  requires_grad = {z_direct.requires_grad}   ← 미분 대상이 아니다")
print(f"  grad_fn = {z_direct.grad_fn}")
print()

# 방식 B: 재파라미터화
eps_one = torch.randn(1)
z_repar = mu2 + sigma2 * eps_one
print(f"[재파라미터화] z = {z_repar.item():.4f}")
print(f"  requires_grad = {z_repar.requires_grad}")
print(f"  grad_fn = {type(z_repar.grad_fn).__name__}   ← 계산 그래프에 연결됨")
print()

z_repar.backward()
print(f"  d z/d mu    = {mu2.grad.item():.4f}")
print(f"  d z/d sigma = {sigma2.grad.item():.4f}  (= 뽑힌 eps)")
print()
print("-" * 60)
print("결과값은 같은 분포에서 나오지만, 미분 가능 여부가 다르다.")
print("무작위성을 eps 쪽으로 밀어내고, 학습 대상은 사칙연산으로만 연결한 것이다.")
print()
print("참고: PyTorch의 rsample() 이 재파라미터화를 자동으로 해 준다.")

---

## 5. VAE 구현 — 이론편 15.4절

이제 VAE를 만든다. AE와 세 가지가 다르다.

| 항목 | AE | VAE |
|---|---|---|
| 인코더 출력 | 잠재 벡터 $z$ | 평균 $\mu$, 로그분산 $\log\sigma^2$ |
| 잠재 벡터 | 그대로 사용 | $z = \mu + \sigma\epsilon$ 로 샘플링 |
| 손실 | 재구성 오차만 | 재구성 오차 + **KL 발산** |

**KL 발산 항**이 잠재 공간을 정규분포에 가깝게 만든다. 3절에서 본 "빈 곳" 문제를 해결하는 것이다.

$$L = \underbrace{\lVert x - \hat{x}\rVert^2}_{\text{잘 복원하라}} + \underbrace{D_{KL}\big(q(z|x)\,\Vert\,\mathcal{N}(0,1)\big)}_{\text{잠재 공간을 정리하라}}$$

In [ ]:
import torch
import torch.nn as nn


class VAE(nn.Module):
    # Variational Autoencoder (이론편 15.4절)

    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128), nn.ReLU(),
        )
        # 평균과 로그분산을 각각 내놓는다
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid(),
        )

    def reparameterize(self, mu, logvar):
        # 이론편 15.4절: z = mu + sigma * eps
        std = torch.exp(0.5 * logvar)     # logvar = log(sigma^2) 이므로
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        out = self.decoder(z).view(-1, 1, 28, 28)
        return out, mu, logvar


def vae_loss(recon, x, mu, logvar):
    # VAE 손실 = 재구성 + KL (이론편 15.4절)
    #
    # KL 항은 정규분포끼리의 경우 닫힌 형태로 계산된다:
    #   KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    recon_loss = nn.functional.binary_cross_entropy(
        recon, x, reduction="sum") / len(x)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / len(x)
    return recon_loss + kl, recon_loss, kl


print("=" * 55)
print("VAE 구조")
print("=" * 55)
vae = VAE(latent_dim=2)
print(f"파라미터: {sum(p.numel() for p in vae.parameters()):,}개")
print()
print("AE와의 차이")
print("  1) 인코더가 mu, logvar 두 개를 내놓는다")
print("  2) z를 샘플링해서 만든다 (재파라미터화)")
print("  3) 손실에 KL 항이 추가된다")
print()
print("왜 logvar(로그분산)를 쓰는가")
print("  sigma는 항상 양수여야 하는데, 신경망 출력은 음수도 나온다.")
print("  log를 씌운 값으로 다루면 exp()로 되돌릴 때 자동으로 양수가 된다.")

In [ ]:
import torch
import time

torch.manual_seed(42)
vae = VAE(latent_dim=2).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

EPOCHS = 15
history_vae = {"total": [], "recon": [], "kl": []}

print("=" * 60)
print("VAE 학습")
print("=" * 60)
t0 = time.time()

for epoch in range(EPOCHS):
    vae.train()
    sums = [0.0, 0.0, 0.0]
    for xb, _ in train_loader:
        xb = xb.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(xb)
        loss, rec, kl = vae_loss(recon, xb, mu, logvar)
        loss.backward()
        optimizer.step()
        for i, v in enumerate([loss, rec, kl]):
            sums[i] += v.item() * len(xb)

    n = len(train_subset)
    history_vae["total"].append(sums[0] / n)
    history_vae["recon"].append(sums[1] / n)
    history_vae["kl"].append(sums[2] / n)

    if (epoch + 1) % 5 == 0:
        print(f"  에폭 {epoch+1:3}: 전체 {sums[0]/n:8.2f} = "
              f"재구성 {sums[1]/n:7.2f} + KL {sums[2]/n:6.2f}  ({time.time()-t0:.0f}초)")

print("-" * 60)
print(f"소요 시간: {time.time()-t0:.0f}초")
print()
print("두 항이 서로 밀고 당긴다:")
print("  재구성 손실 ↓ 를 원하면 각 데이터를 구별되게 흩어놓아야 한다")
print("  KL ↓ 를 원하면 모두 원점 근처로 모아야 한다")
print("  이 균형점에서 '빈 곳 없이 채워진' 잠재 공간이 만들어진다.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# AE와 VAE의 잠재 공간 비교
vae.eval()
zs_vae, labels_vae = [], []
with torch.no_grad():
    for i in range(0, 2000, 256):
        batch = torch.stack([test_data[j][0] for j in range(i, min(i+256, 2000))]).to(device)
        lbl = [test_data[j][1] for j in range(i, min(i+256, 2000))]
        _, mu, _ = vae(batch)
        zs_vae.append(mu.cpu().numpy())
        labels_vae.extend(lbl)

Z_vae = np.concatenate(zs_vae)
labels_vae = np.array(labels_vae)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (Zd, lbl, title) in zip(axes, [
        (Z, labels, "Autoencoder"),
        (Z_vae, labels_vae, "VAE")]):
    sc = ax.scatter(Zd[:, 0], Zd[:, 1], c=lbl, cmap="tab10", s=8, alpha=0.6)
    ax.set_title(f"{title}의 잠재 공간")
    ax.set_xlabel("잠재 차원 1")
    ax.set_ylabel("잠재 차원 2")
    ax.grid(alpha=0.3)

plt.colorbar(sc, ax=axes, ticks=range(10), label="클래스")
plt.show()

print("=" * 55)
print("잠재 공간 비교")
print("=" * 55)
GRID = 12
for name, Zd in [("AE", Z), ("VAE", Z_vae)]:
    h, _, _ = np.histogram2d(Zd[:, 0], Zd[:, 1], bins=GRID)
    empty = (h == 0).sum()
    print(f"{name:<6} 범위 {Zd.min():6.2f}~{Zd.max():6.2f}  "
          f"빈 칸 {empty}/{GRID*GRID} ({empty/(GRID*GRID)*100:.0f}%)")
print()
print("VAE 쪽이 원점 주변에 고르게 모여 있다. KL 항이 그렇게 만든 것이다.")
print("→ 이제 무작위로 뽑아도 그럴듯한 결과가 나온다 (6절).")

---

## 6. 생성과 보간 — 이론편 15.5절

VAE의 잠재 공간은 정규분포에 가깝게 정리되었다. **정규분포에서 뽑아 디코더에 넣으면**
새로운 이미지가 나온다.

In [ ]:
import torch
import matplotlib.pyplot as plt

vae.eval()

# 표준정규분포에서 뽑아 생성
torch.manual_seed(7)
with torch.no_grad():
    z_sample = torch.randn(16, 2).to(device)
    generated = vae.decoder(z_sample).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i], cmap="gray")
    ax.axis("off")
fig.suptitle("VAE가 생성한 이미지 (정규분포에서 무작위 샘플링)", fontsize=12)
plt.tight_layout()
plt.show()

print("3절의 AE 결과와 비교해 보라.")
print("대부분이 옷 형태를 갖추고 있다 — 잠재 공간이 정리되었기 때문이다.")
print()
print("이것이 '생성 모델'이라 부르는 이유다.")
print("학습 데이터에 없던 것을 만들어 낸다 (이론편 15.5절).")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 잠재 공간 격자를 훑으며 디코딩
GRID = 12
vae.eval()

# 표준정규분포의 분위수로 격자를 만든다
from scipy.stats import norm
grid_x = norm.ppf(np.linspace(0.02, 0.98, GRID))
grid_y = norm.ppf(np.linspace(0.02, 0.98, GRID))

canvas = np.zeros((GRID * 28, GRID * 28))
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            img = vae.decoder(z).view(28, 28).cpu().numpy()
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(canvas, cmap="gray")
ax.set_title("잠재 공간 전체를 훑어 본 결과", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

print("한 칸씩 옮길 때마다 모양이 조금씩 변한다.")
print("바지 → 옷 → 신발처럼 연속적으로 이어지는 구간이 보인다.")
print()
print("잠재 공간이 '빈 곳 없이' 채워졌다는 증거다 (이론편 15.5절).")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 두 이미지 사이를 보간 (이론편 15.5절)
vae.eval()

idx_a, idx_b = 0, 1
img_a = test_data[idx_a][0].unsqueeze(0).to(device)
img_b = test_data[idx_b][0].unsqueeze(0).to(device)

with torch.no_grad():
    _, mu_a, _ = vae(img_a)
    _, mu_b, _ = vae(img_b)

    steps = 9
    alphas = np.linspace(0, 1, steps)
    interpolated = []
    for a in alphas:
        z = (1 - a) * mu_a + a * mu_b       # 잠재 공간에서 직선 이동
        interpolated.append(vae.decoder(z).view(28, 28).cpu().numpy())

fig, axes = plt.subplots(2, steps, figsize=(14, 3.5))

# 위: 잠재 공간 보간
for i, (a, img) in enumerate(zip(alphas, interpolated)):
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(f"{a:.2f}", fontsize=8)
    axes[0, i].axis("off")

# 아래: 픽셀 공간에서 그냥 섞기 (비교용)
img_a_np = img_a.cpu().squeeze().numpy()
img_b_np = img_b.cpu().squeeze().numpy()
for i, a in enumerate(alphas):
    axes[1, i].imshow((1 - a) * img_a_np + a * img_b_np, cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("잠재 공간", fontsize=9)
fig.text(0.02, 0.72, "잠재 공간\n보간", fontsize=10, va="center")
fig.text(0.02, 0.28, "픽셀 직접\n섞기", fontsize=10, va="center")
plt.tight_layout(rect=[0.06, 0, 1, 1])
plt.show()

print(f"왼쪽: {test_data.classes[test_data[idx_a][1]]}  →  "
      f"오른쪽: {test_data.classes[test_data[idx_b][1]]}")
print()
print("위: 각 단계가 '있을 법한 옷'의 모습을 유지한다")
print("아래: 두 이미지가 겹쳐 보이는 유령 같은 이미지가 된다")
print()
print("차이가 나는 이유 — 잠재 공간은 '의미'의 공간이기 때문이다.")
print("두 점 사이를 지날 때도 의미 있는 지점들을 통과한다 (이론편 15.5절).")

---

## 7. 흐릿함의 원인 — 이론편 15.6절

VAE로 생성한 이미지를 보면 **원본보다 흐릿하다.** 이론편 15.6절에서 다룬 이 현상의 원인을 확인한다.

**원인은 손실함수에 있다.** 픽셀 단위 오차를 최소화하면, 확신이 없을 때
**평균값을 내놓는 것이 안전**하다.

예를 들어 어떤 위치에 무늬가 있을 수도 없을 수도 있다면, 중간 밝기로 칠하는 것이
평균적으로 오차가 작다. 이것이 반복되면 전체가 흐릿해진다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

vae.eval()
model_ae.eval()

with torch.no_grad():
    sample = torch.stack([test_data[i][0] for i in range(6)]).to(device)
    recon_ae, _ = model_ae(sample)
    recon_vae, _, _ = vae(sample)

fig, axes = plt.subplots(3, 6, figsize=(11, 5.5))
titles = ["원본", "AE 복원", "VAE 복원"]
for row, imgs in enumerate([sample, recon_ae, recon_vae]):
    for col in range(6):
        axes[row, col].imshow(imgs[col].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_title(titles[row], loc="left", fontsize=10)
plt.tight_layout()
plt.show()

# 선명도를 숫자로 — 인접 픽셀 차이의 크기
def sharpness(imgs):
    x = imgs.cpu().squeeze().numpy()
    dx = np.abs(np.diff(x, axis=-1)).mean()
    dy = np.abs(np.diff(x, axis=-2)).mean()
    return (dx + dy) / 2

print("=" * 55)
print("선명도 비교 (인접 픽셀 차이의 평균)")
print("=" * 55)
s_orig = sharpness(sample)
for name, imgs in [("원본", sample), ("AE 복원", recon_ae), ("VAE 복원", recon_vae)]:
    s = sharpness(imgs)
    print(f"  {name:<12}{s:.4f}   (원본 대비 {s/s_orig*100:.0f}%)")
print("-" * 55)
print()
print("복원 이미지가 원본보다 변화가 완만하다 = 흐릿하다.")
print()
print("이 문제를 다른 방식으로 접근한 것이 다음 장의 GAN이다.")
print("픽셀 오차 대신 '진짜처럼 보이는가'를 판별기가 평가한다 (이론편 16.1절).")

---

## 8. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **15.4** | **재파라미터화 z = 1.50, 2.00, 2.25, 2.75** | **일치** ✓ |
| 15.4 | 미분 가능성 (dz/dmu = 1) | 확인 ✓ |
| 15.3 | AE로는 생성 불가 | 빈 잠재 공간 확인 ✓ |
| 15.5 | 잠재 공간 보간 | 픽셀 섞기와 대비 ✓ |
| 15.6 | 생성물이 흐릿함 | 선명도 수치로 확인 ✓ |

### AE와 VAE의 차이

| | AE | VAE |
|---|---|---|
| 인코더 출력 | 점 하나 | 분포 (mu, logvar) |
| 잠재 공간 | 빈 곳이 많음 | 정규분포에 가깝게 정리 |
| 생성 | 불가 | 가능 |
| 손실 | 재구성만 | 재구성 + KL |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 재파라미터화 | 무작위성을 eps로 분리 → 미분 가능 |
| `logvar` 사용 | 신경망 출력이 음수여도 exp로 양수 보장 |
| KL 항 | 잠재 공간을 정규분포 쪽으로 당김 |
| 두 손실의 균형 | 재구성은 흩어놓으려 하고 KL은 모으려 함 |
| 흐릿함 | 픽셀 오차 최소화 → 애매하면 평균값 |

### 다음 장

**19. GAN과 Diffusion — 흐릿함을 넘어서** — 이론편 20장. 흐릿함 문제를 다른 방식으로 푼 두 접근을 다룬다.
이론편 16.3절의 노이즈 누적 표를 직접 계산해 확인한다.